<a href="https://colab.research.google.com/github/YifeiCAO/CogModelingRNNsTutorial/blob/train/three_arm_vrnn_cog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import and preparation

In [ ]:
AS_OUTER_LOOP = [1,2,3,4,5,6,7,8,9,10] # assigned outer loop(s) to run in this notebook

In [ ]:
!pip -q install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 94.8 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd
from io import StringIO
import gdown

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import numpy as np
import pandas as pd
# import plotnine as gg
# gg.theme_set(gg.theme_classic)  # for nicer-looking plots
import jax.numpy as jnp
import jax
import optax
import scipy

!pip install -U dm-haiku
!pip install chex
import haiku as hk
rng_seq = hk.PRNGSequence(np.random.randint(2**32))

#@title Install required packages.
try:
    from google.colab import files
    _ON_COLAB = True
except:
    _ON_COLAB = False

if _ON_COLAB:
  !rm -rf CogModelingRNNsTutorial
  !git clone https://github.com/YifeiCAO/CogModelingRNNsTutorial
  !pip install -e CogModelingRNNsTutorial/CogModelingRNNsTutorial
  !cp CogModelingRNNsTutorial/CogModelingRNNsTutorial/*py CogModelingRNNsTutorial
else:
  !pip install CogModelingRNNsTutorial/requirements.txt

#@title Imports + defaults settings.
# %load_ext autoreload
# %autoreload 2
# for reload
# %reload_ext autoreload

# import haiku as hk
# import jax
# import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optax
import os
import warnings

# warnings.filterwarnings("ignore")

# try:
#     from google.colab import files
#     _ON_COLAB = True
# except:
#     _ON_COLAB = False

from CogModelingRNNsTutorial import bandits
from CogModelingRNNsTutorial import disrnn
from CogModelingRNNsTutorial import hybrnn
from CogModelingRNNsTutorial import hybconrnn
from CogModelingRNNsTutorial import hybrnn_direct_con
from CogModelingRNNsTutorial import plotting
from CogModelingRNNsTutorial import rat_data
from CogModelingRNNsTutorial import rnn_utils

from google.colab import drive
import pickle

# 挂载 Google Drive
drive.mount('/content/drive')

Cloning into 'CogModelingRNNsTutorial'...
remote: Enumerating objects: 1379, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 1379 (delta 213), reused 192 (delta 161), pack-reused 1112 (from 1)
Receiving objects: 100% (1379/1379), 6.88 MiB | 33.41 MiB/s, done.
Resolving deltas: 100% (886/886), done.
Obtaining file:///content/CogModelingRNNsTutorial/CogModelingRNNsTutorial
  Preparing metadata (setup.py) ... done
  Running setup.py develop for CogModelingRNNsTutorial
Mounted at /content/drive


In [ ]:
def compute_negative_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]

    # 直接在外部运行 eval_model（不要放 jit 里！）
    model_outputs, _ = rnn_utils.eval_model(model_fun, params, xs)

    # 再 jit 一个小函数，只对 model_outputs 做 softmax
    @jax.jit
    def compute_log_probs(logits):
        return jax.nn.log_softmax(logits[:, :, :2], axis=-1)

    predicted_log_choice_probabilities = compute_log_probs(model_outputs)

    # 后续仍转为 NumPy 做索引
    predicted_log_choice_probabilities = np.array(predicted_log_choice_probabilities)

    total_log_likelihood = 0.0
    total_valid_trials = 0

    for sess_i in range(n_sessions):
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            if actual_choice >= 0:
                total_log_likelihood += predicted_log_choice_probabilities[trial_i, sess_i, actual_choice]
                total_valid_trials += 1

    if total_valid_trials == 0:
        raise ValueError("No valid trials found.")

    avg_nll = -total_log_likelihood / total_valid_trials
    return avg_nll



def evaluate_nll_over_full_dataset(dataset, model_fun, params):
    all_nlls = []
    for _ in range(dataset.n_batches):
        nll = compute_negative_log_likelihood(dataset, model_fun, params)
        all_nlls.append(nll)

    avg_nll = np.mean(all_nlls)
    print(f"[All Batches Averaged] NLL: {avg_nll:.4f}")
    return avg_nll


In [ ]:
@jax.jit
def compute_log_likelihood(dataset, model_fun, params):
    xs, actual_choices = next(dataset)
    n_trials_per_session, n_sessions = actual_choices.shape[:2]
    model_outputs, model_states = rnn_utils.eval_model(model_fun, params, xs)

    # predicted log-probs for the first two actions
    predicted_log_choice_probabilities = np.array(
        jax.nn.log_softmax(model_outputs[:, :, :2], axis=-1)
    )

    n_actions = predicted_log_choice_probabilities.shape[2]
    log_likelihoods = []

    for sess_i in range(n_sessions):
        log_likelihood = 0.0
        n = 0
        for trial_i in range(n_trials_per_session):
            actual_choice = int(actual_choices[trial_i, sess_i])
            # ignore invalid trials (<0 or ≥n_actions)
            if 0 <= actual_choice < n_actions:
                log_likelihood += predicted_log_choice_probabilities[
                    trial_i, sess_i, actual_choice
                ]
                n += 1

        if n > 0:
            normalized_likelihood = np.exp(log_likelihood / n)
            log_likelihoods.append(normalized_likelihood)

    mean_likelihood = np.mean(log_likelihoods)
    std_likelihood  = np.std(log_likelihoods)

    print(f'Average Normalized Likelihood: {100 * mean_likelihood:.1f}%')
    return mean_likelihood, std_likelihood


In [ ]:
jax.devices()

[CpuDevice(id=0)]

## Human Datas

In [ ]:
osf_url = 'https://osf.io/download/xe6yu/'
response = requests.get(osf_url)

# Check the request
if response.status_code == 200:
    # Read as pandas dataframe
    qasim_data = pd.read_csv(StringIO(response.text))
    print('File downloaded and read successfully!')
else:
    print('Failed to download file. Status code:', response.status_code)

qasim_data.head()

# # read data
selected_columns = ['participant', 'trials_gamble', 'gamble', 'prob', 'reward']

qasim = qasim_data[selected_columns]
qasim_filtered = qasim[qasim['trials_gamble'].notna()]
qasim_sorted = qasim_filtered.groupby('participant', group_keys=False).apply(lambda x: x.sort_values('trials_gamble'))
qasim_sorted = qasim_sorted.reset_index(drop=True)
qasim_sorted['participant'] = qasim_sorted.groupby(['participant']).ngroup() + 1
qasim_sorted['action'] = qasim_sorted['gamble']
qasim_sorted

# —— 2) 把缺失的 action 填成 -1 —— #
qasim_sorted['action'] = qasim_sorted['action'].fillna(-1).astype(int)

# —— 3) 如果需要，把 reward 映射到 0/1 —— #
# （如果已经是 0/1 可跳过；否则取消下面注释并调整映射字典）
# qasim_sorted['reward'] = (
#     qasim_sorted['reward']
#     .map({-1: 0, 1: 1})
#     .fillna(-1)
#     .astype(int)
# )

# —— 4) 排序，确保 trial 顺序 —— #
qasim_sorted = qasim_sorted.sort_values(
    ['participant', 'trials_gamble']
).reset_index(drop=True)

# —— 5) 生成下一步动作 action_n —— #
qasim_sorted['action_n'] = (
    qasim_sorted
    .groupby('participant')['action']
    .shift(-1)
)
# 每个 participant 最后一 trial 的 action_n 设为 -1
last_idxs = qasim_sorted.groupby('participant').tail(1).index
qasim_sorted.loc[last_idxs, 'action_n'] = -1

# —— 6) 按 participant 构造 xs_list, ys_list —— #
xs_list, ys_list = [], []
for pid, grp in qasim_sorted.groupby('participant'):
    grp = grp.sort_values('trials_gamble')
    x = grp[['action', 'reward']].to_numpy().astype(float)    # 输入特征
    y = grp[['action_n']].to_numpy().astype(int)             # 下一步动作
    xs_list.append(x)
    ys_list.append(y)

# —— 7) 堆成三维数组 —— #
xs_qa = np.stack(xs_list, axis=1)  # (n_sessions, n_trials, 2)
ys_qa = np.stack(ys_list, axis=1)  # (n_sessions, n_trials, 1)

print("xs.shape =", xs_qa.shape)
print("ys.shape =", ys_qa.shape)



### Sidarus dataset

In [ ]:
# # 修改后的 file_id
# file_id = '1TSV6CdyClKz831qD2ln4z3WjLLcOgG1n'
# download_url = f'https://drive.google.com/uc?id={file_id}'

# # 下载并保存为 'downloaded_file.csv'
# output_file = 'downloaded_file.csv'
# gdown.download(download_url, output_file, quiet=False)

# # 读取 CSV 数据
# sida_data = pd.read_csv(output_file)
# sida_data

# # 1) action 从 {1,2} → {0,1}，并把所有缺失值填成 -1
# sida_data['action'] = (
#     sida_data['action']
#     .map({1: 0, 2: 1})    # 映射
#     .fillna(-1)           # 缺失值设为 -1
#     .astype(int)
# )

# # 2) outcome 从 {-1,1} → {0,1}，缺失也填成 -1
# sida_data['reward'] = (
#     sida_data['outcome']
#     .map({-1: 0, 1: 1})
#     .fillna(-1)
#     .astype(int)
# )

# # 3) 给每个被试×session 分配一个连续的 session_id
# sida_data['session_id'] = (
#     sida_data
#     .groupby(['subj', 'session'], sort=False)
#     .ngroup()
#     + 1
# )

# # 4) 排序，保证 trial 顺序
# sida_data = sida_data.sort_values(
#     ['session_id', 'epN', 'epTrialN']
# ).reset_index(drop=True)

# # 5) 生成下一步动作 action_n；每个 session 末尾设为 -1
# sida_data['action_n'] = (
#     sida_data
#     .groupby('session_id')['action']
#     .shift(-1)
# )
# last_idx = sida_data.groupby('session_id').tail(1).index
# sida_data.loc[last_idx, 'action_n'] = -1

# # 6) 按 session_id 抽取序列，并堆成 xs, ys
# xs_list, ys_list = [], []
# for sid, grp in sida_data.groupby('session_id'):
#     grp = grp.sort_values(['epN', 'epTrialN'])
#     x = grp[['action', 'reward']].to_numpy().astype(float)  # 特征
#     y = grp[['action_n']].to_numpy().astype(int)              # Label
#     xs_list.append(x)
#     ys_list.append(y)

# # 最终结果：xs.shape == (n_sessions, n_trials, 2)，ys.shape == (n_sessions, n_trials, 1)
# xs_sida = np.stack(xs_list, axis=1)
# ys_sida = np.stack(ys_list, axis=1)

# print("xs.shape =", xs_sida.shape)
# print("ys.shape =", ys_sida.shape)


In [ ]:
import gdown
import pandas as pd

# 替换为你刚才发的链接中的 file_id
file_id = '1FvuBsn25uqaRdcuB4Dpxc5fx8cQ1Pk30'
download_url = f'https://drive.google.com/uc?id={file_id}'

# 下载并保存为 'downloaded_file.csv'
output_file = 'downloaded_file.csv'
gdown.download(download_url, output_file, quiet=False)

# 读取 CSV 数据
sahar_data = pd.read_csv(output_file)

In [ ]:
# 1. 提取前 500 行
# head(500) 会自动选取前 500 行数据
subset_data = sahar_data.head(500)

# 2. 设置保存路径
# 注意：Drive 挂载后的标准根目录通常是 /content/drive/MyDrive/
# 你可以根据需要修改文件名
save_path = '/content/drive/MyDrive/sahar_data_top500.csv'

# 3. 保存为 CSV
# index=False 表示不把 pandas 自动生成的 0,1,2... 索引列存进去
subset_data.to_csv(save_path, index=False)

print(f"成功！文件已保存到: {save_path}")

### Schaaf dataset

In [ ]:
# 修改后的 file_id
file_id = '1rJFmDhCE3fdXtSHSSvtre_7V49gGsg7P'
download_url = f'https://drive.google.com/uc?id={file_id}'

# 下载并保存为 'downloaded_file.csv'
output_file = 'downloaded_file.csv'
gdown.download(download_url, output_file, quiet=False)

# 读取 CSV 数据
schaaf_data = pd.read_csv(output_file)
schaaf_data

df1 = schaaf_data[['pp', 'trial1', 'response1', 'outcome1', 'condition1']].copy()
df1.columns = ['pp', 'trial_in_session', 'response', 'reward', 'condition']
df1['session'] = 1

df2 = schaaf_data[['pp', 'trial2', 'response2', 'outcome2', 'condition2']].copy()
df2.columns = ['pp', 'trial_in_session', 'response', 'reward', 'condition']
df2['session'] = 2

# —— 上面拆分、concat、sort 的部分保持不变 —— #

# 合并成长表
df_long = pd.concat([df1, df2], ignore_index=True)
df_long = df_long.sort_values(['pp', 'session', 'trial_in_session']).reset_index(drop=True)

# —— 映射 outcome，再填 missing response —— #
# 把原来的 response1/response2 改名后是 `response`
# 把 outcome1/outcome2 改名后是 `reward`
# 把 outcome 映射成 1/0，然后把原本缺失的 reward 补成 -1
df_long['reward'] = (
    df_long['reward']
    .map({1: 1, -1: 0})
    .fillna(-1)
    .astype(int)
)

# 把缺失的 response 一样补成 -1
df_long['response'] = (
    df_long['response']
    .fillna(-1)
    .astype(int)
)


# 接下来再做 session_id、shift action_n、以及 stack xs/ys 的流程……
df_long['session_id'] = (df_long['pp'] - 1) * 2 + df_long['session']
df_long['action_n'] = df_long.groupby('session_id')['response'].shift(-1)
last_idx = df_long.groupby('session_id').tail(1).index
df_long.loc[last_idx, 'action_n'] = -1

# 生成 xs, ys
session_ids = df_long['session_id'].unique()
xs_list, ys_list = [], []
for sid in session_ids:
    sd = df_long[df_long['session_id'] == sid].sort_values('trial_in_session')
    x = sd[['response', 'reward']].to_numpy().astype(float)
    y = sd[['action_n']].to_numpy().astype(int)
    xs_list.append(x)
    ys_list.append(y)

xs = np.stack(xs_list, axis=0)   # (n_sessions, n_trials, 2)
ys = np.stack(ys_list, axis=0)   # (n_sessions, n_trials, 1)


# 或者第 0 维是 trials，第 1 维是 sessions：
xs_sch = np.stack(xs_list, axis=1)  # (n_trials, n_sessions, 2)
ys_sch = np.stack(ys_list, axis=1)  # (n_trials, n_sessions, 1)

print("xs.shape =", xs_sch.shape)
print("ys.shape =", ys_sch.shape)

# 1 → 0
# 方法一：一次性用 np.where 嵌套
xs_sch[:, :, 0] = np.where(
    xs_sch[:, :, 0] == 2,  # 如果等于 2
    1,                         # 则变成 1
    np.where(
        xs_sch[:, :, 0] == 1,  # 否则如果等于 1
        0,                         # 则变成 0
        xs_sch[:, :, 0]        # 其它值保持原样
    )
)
ys_sch[ys_sch == 1] = 0
ys_sch[ys_sch == 2] = 1

In [ ]:
df_long

In [ ]:
xs_sch.shape

In [ ]:
switch_num_sch = np.ones((xs_sch.shape[0], xs_sch.shape[1])) * 999
for sid in range(xs_sch.shape[1]):
  for tid in range(1, xs_sch.shape[0]):
    cond = (df_long.set_index(["session_id", "trial_in_session"]).loc[(sid+1, tid+1), "condition"])
    cond_prev = (df_long.set_index(["session_id", "trial_in_session"]).loc[(sid+1, tid), "condition"])
    if cond != cond_prev:
      switch_num_sch[tid, sid] = 0
      for offset in range(1, 8):
        if tid - offset >= 0:
          switch_num_sch[tid-offset, sid] = -offset
      for offset in range(1,11):
        if tid + offset < xs.shape[0]:
          switch_num_sch[tid+offset, sid] = offset
switch_num_sch


In [ ]:
# xs_flat = xs_sch[:-1, :, 1]  # 或用 ys[:,:,0]
# # 1) 有效 trial 的掩码（0 或 1 才算），shape (n_trials, n_sessions)
# valid = xs_flat >= 0
# # 2) 每个 session 成功的次数（sum of 1s）
# n_success = np.sum(xs_flat * valid, axis=0)
# # 3) 每个 session 的有效 trial 数
# n_valid   = np.sum(valid, axis=0)
# # 4) 正确率 = 成功次数 / 有效数
# accuracy_per_session = n_success / n_valid

# # 打印
# for i, acc in enumerate(accuracy_per_session):
#     print(f"Session {i:02d}: accuracy = {acc:.4f} ({acc*100:.2f}%)")

### Maria Dataset

In [ ]:
import pandas as pd
import numpy as np
import gdown

# —— 1) 下载并读入原始数据 —— #
file_id = '1N_zAy-qrbfjvF8Kbb504IH2JNhR5KI-P'
url     = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, 'downloaded_file.csv', quiet=False)
eck_data = pd.read_csv('downloaded_file.csv')

# —— 2) 筛选＆重命名列 —— #
sel = ['sID','TrialID','selected_box','reward', 'block']
eck_sorted = eck_data[sel].copy()
eck_sorted['participant'] = eck_sorted.groupby('sID').ngroup()+1
eck_sorted['action']      = eck_sorted['selected_box']

# 只保留至少做够 120 试次的被试
max_trial = eck_sorted.groupby('participant')['TrialID'].transform('max')
eck_sorted = eck_sorted[max_trial>=120]

# 只用前 120 试次
eck_sorted = eck_sorted[eck_sorted['TrialID']<=120].reset_index(drop=True)

# —— 3) 生成下一步动作 action_n —— #
def generate_action_n(group):
    group = group.sort_values('TrialID')
    group['action_n'] = group['action'].shift(-1).fillna(-1).astype(int)
    return group

eck_sorted = (
    eck_sorted
    .groupby('participant', group_keys=False)
    .apply(generate_action_n)
    .reset_index(drop=True)
)

# —— 4) 按 participant 构造 xs_list/ys_list —— #
xs_list, ys_list = [], []

for pid, grp in eck_sorted.groupby('participant'):
    grp = grp.sort_values('TrialID').iloc[:120]
    x = grp[['action','reward']].to_numpy().astype(float)  # (120,2)
    y = grp[['action_n']].to_numpy().astype(int)           # (120,1)
    xs_list.append(x)
    ys_list.append(y)

# —— 5) stack 成 (n_sessions, n_trials, feat_dim) —— #
xs_ma = np.stack(xs_list, axis=1)  # (305,120,2)
ys_ma = np.stack(ys_list, axis=1)  # (305,120,1)

print("xs_ma.shape =", xs_ma.shape)
print("ys_ma.shape =", ys_ma.shape)


In [ ]:
eck_sorted

In [ ]:
# switch_num_ma = np.ones((xs_ma.shape[0], xs_ma.shape[1])) * 999
# for sid in range(xs_ma.shape[1]):
#   for tid in range(1, xs_ma.shape[0]):
#     cond = (eck_sorted.set_index(["sID", "TrialID"]).loc[(sid+1, tid+1), "block"])
#     cond_prev = (eck_sorted.set_index(["sTD", "TrialID"]).loc[(sid+1, tid), "block"])
#     if cond != cond_prev:
#       switch_num_ma[tid, sid] = 0
#       for offset in range(1, 8):
#         if tid - offset >= 0:
#           switch_num_ma[tid-offset, sid] = -offset
#       for offset in range(1,11):
#         if tid + offset < xs.shape[0]:
#           switch_num_ma[tid+offset, sid] = offset
# switch_num_ma


## Clean Data

## Generate a big human dataset with all experiments, session length is 130 trial per session

In [ ]:
import numpy as np

def segment_and_pad(x: np.ndarray,
                    y: np.ndarray,
                    seg_len: int = 130,
                    pad_x: float = 0.,
                    pad_y: int = -1):
    T, D = x.shape
    n_segs = int(np.ceil(T / seg_len))
    x_segs, y_segs = [], []
    for i in range(n_segs):
        start = i * seg_len
        end = start + seg_len
        x_part = x[start : min(end, T)]
        y_part = y[start : min(end, T)]
        pad = end - min(end, T)
        if pad > 0:
            x_part = np.pad(x_part,
                            pad_width=((0, pad), (0, 0)),
                            constant_values=pad_x)
            y_part = np.pad(y_part,
                            pad_width=((0, pad), (0, 0)),
                            constant_values=pad_y)
        x_segs.append(x_part)
        y_segs.append(y_part)
    return x_segs, y_segs

# —— 假设你已经有这几组 (xs, ys) —— #
# xs_qa   (60,  206, 2), ys_qa  (60,206, 1)
# xs_sida (800, 40,  2), ys_sida(800,40, 1)
# xs_sch  (249, 44,  2), ys_sch (249,44, 1)
# xs_ma   (120,305, 2), ys_ma  (120,305,1)

all_xs, all_ys = [], []

for xs, ys in [(xs_qa, ys_qa),
               (xs_sch, ys_sch),
               (xs_ma, ys_ma)]:

    # 如果是 (n_trials, n_sessions, feat) 维度，就直接：
    # T, N, D = xs.shape
    # 否则若是 (n_sessions, n_trials, feat)，先转：
    # xs = xs.transpose(1,0,2)
    # ys = ys.transpose(1,0,2)

    T, N, D = xs.shape
    for sess in range(N):
        x_seq = xs[:, sess, :]  # (T, D)
        y_seq = ys[:, sess, :]  # (T, 1)
        x_segs, y_segs = segment_and_pad(
            x_seq, y_seq,
            seg_len=130,
            pad_x=0., pad_y=-1
        )
        all_xs.extend(x_segs)
        all_ys.extend(y_segs)

# —— 修改在这里：不要 np.stack，直接输出列表 —— #
# all_xs 是一个 Python list，长度 = 总片段数，每个元素 shape=(130, D)
# all_ys 是一个 Python list，长度 = 总片段数，每个元素 shape=(130, 1)

print(f"Total segments: {len(all_xs)}")
print(f"First xs segment shape: {all_xs[0].shape}")
print(f"First ys segment shape: {all_ys[0].shape}")

# 如果你需要把它们返回成变量：
xs_segment_list = all_xs
ys_segment_list = all_ys


In [ ]:
def format_into_datasets_10_multi(
    xs_list: list[np.ndarray],
    ys_list: list[np.ndarray],
    dataset_constructor,
    batch_size: int = None,
    random_seed: int = None,
):
    """
    对多只猴子独立做 10‐fold CV，每折：
      - test = 每只猴子第 i 份 fold
      - validation 抽取与 test 大小一致的 session
      - 剩余做 train
    Args:
      xs_list: list of np.ndarray, 每项形状 (timesteps, n_sess_i, feat)
      ys_list: list of np.ndarray, 每项形状 (timesteps, n_sess_i, ...)
      dataset_constructor: 接受 (xs_sub, ys_sub, batch_size) 的构造器
      batch_size: mini‐batch 大小；None 则每 dataset 用全部 episodes
      random_seed: int，可复现打乱
    Returns:
      List of 10 tuples (ds_train, ds_val, ds_test)
    """
    rng = np.random.RandomState(random_seed) if random_seed is not None else np.random

    n_monkeys = len(xs_list)
    # 1. 对每只猴子生成打乱后的 session 索引，并切成 10 份
    folds_per_monkey = []
    for xs in xs_list:
        n_sess = xs.shape[1]
        idx = rng.permutation(n_sess)
        folds = np.array_split(idx, 10)
        folds_per_monkey.append(folds)

    folds_datasets = []
    for i in range(10):
        # 2a. 测试集：三猴子各自的第 i 份
        test_parts = [folds_per_monkey[m][i] for m in range(n_monkeys)]

        # 2b. 剩余 sessions 用于 train+val
        rem_parts = [
            np.hstack([folds_per_monkey[m][j] for j in range(10) if j != i])
            for m in range(n_monkeys)
        ]

        # 2c. 验证集大小与测试集一致：每只猴子抽取与 test_parts[m] 相同数量
        val_parts = []
        train_parts = []
        for m in range(n_monkeys):
            rng.shuffle(rem_parts[m])
            n_test = len(test_parts[m])
            val = rem_parts[m][:n_test]
            train = rem_parts[m][n_test:]
            val_parts.append(val)
            train_parts.append(train)

        # 3. 拼接各猴子的 train/val/test xs, ys
        def _gather(parts_idx):
            xs_sub, ys_sub = [], []
            for m, idxs in enumerate(parts_idx):
                xs_sub.append(xs_list[m][:, idxs, :])
                ys_sub.append(ys_list[m][:, idxs, :])
            return np.concatenate(xs_sub, axis=1), np.concatenate(ys_sub, axis=1)

        xs_train, ys_train = _gather(train_parts)
        xs_val,   ys_val   = _gather(val_parts)
        xs_test,  ys_test  = _gather(test_parts)

        # 4. 构造 DatasetRNN
        ds_train = dataset_constructor(xs_train, ys_train, batch_size=batch_size)
        ds_val   = dataset_constructor(xs_val,   ys_val,   batch_size=batch_size)
        ds_test  = dataset_constructor(xs_test,  ys_test,  batch_size=batch_size)

        folds_datasets.append((ds_train, ds_val, ds_test))

    return folds_datasets


In [ ]:
# 假设你已经有原始的：
#   xs_qa   (T_qa,   N_qa,   D),   ys_qa   (T_qa,   N_qa,   1)
#   xs_sida (T_sida, N_sida, D),   ys_sida (T_sida, N_sida, 1)
#   xs_sch  (T_sch,  N_sch,  D),   ys_sch  (T_sch,  N_sch,  1)
#   xs_ma   (T_ma,   N_ma,   D),   ys_ma   (T_ma,   N_ma,   1)

def make_segmented_array(xs, ys, seg_len=130, pad_x=0., pad_y=-1):
    all_xs, all_ys = [], []
    T, N, D = xs.shape
    for sess in range(N):
        x_seq = xs[:, sess, :]    # (T, D)
        y_seq = ys[:, sess, :]    # (T, 1)
        x_segs, y_segs = segment_and_pad(x_seq, y_seq, seg_len, pad_x, pad_y)
        all_xs.extend(x_segs)     # list of (130, D)
        all_ys.extend(y_segs)     # list of (130, 1)
    # 把 list 再拼成一个三维 array (130, n_segments, D)
    xs_seg = np.stack(all_xs, axis=1)
    ys_seg = np.stack(all_ys, axis=1)
    return xs_seg, ys_seg

# 针对四个源分别做一次
xs_qa_seg,   ys_qa_seg   = make_segmented_array(xs_qa,   ys_qa)
xs_sch_seg, ys_sch_seg  = make_segmented_array(xs_sch,  ys_sch)
xs_ma_seg,  ys_ma_seg   = make_segmented_array(xs_ma,   ys_ma)

In [ ]:
xs_qa_seg.shape, ys_qa_seg.shape

In [ ]:
xs_human_list = [xs_qa_seg, xs_sch_seg, xs_ma_seg]
ys_human_list = [ys_qa_seg, ys_sch_seg, ys_ma_seg]

### New Dataset

In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX'
download_url = f'https://drive.google.com/uc?id={file_id}'

# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
suthana_data = pd.read_csv(output_file)

# Display the first few rows
suthana_data.head()

Downloading...
From: https://drive.google.com/uc?id=1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX
To: /content/downloaded_new_file.csv
100%|██████████| 2.05M/2.05M [00:00<00:00, 34.5MB/s]


,subject,session,action,reward,switch
0,TP463,1,1,1,0
1,TP463,1,3,1,0
2,TP463,1,1,1,0
3,TP463,1,3,1,0
4,TP463,1,3,1,0


In [ ]:
suthana_data.shape

(150080, 5)

In [ ]:
# 1. Assign a unique numeric participant ID
# We can use the 'subject' column to group and assign IDs.
suthana_data['participant_id'] = suthana_data.groupby('subject').ngroup()

# 2. Sort by participant and session/trial order
# Assuming the original order is correct, but good to be safe if there were trial numbers.
# Since there is no trial column, we rely on the existing order, but ensure we process by participant.
# We can just use the index if the data is already sorted by trial within subject.

# 3. Generate next action 'action_n'
# We group by participant and shift the 'action' column by -1.
suthana_data['action_n'] = suthana_data.groupby('participant_id')['action'].shift(-1)

# Fill the last trial's next action with -1 for each participant
# (The shift operation produces NaN at the end of each group)
suthana_data['action_n'] = suthana_data['action_n'].fillna(-1).astype(int)

# 4. Construct xs and ys lists
xs_list_suth = []
ys_list_suth = []

# Segment length
seg_len = 160

# Iterate through each participant
for pid, grp in suthana_data.groupby('participant_id', sort=True):
    # Get total trials for this subject
    total_trials = len(grp)

    # Calculate how many full 160-trial segments we can make
    n_segments = total_trials // seg_len

    if n_segments == 0:
        # Skip if less than 160 trials (or handle as padding if you preferred, but 'cut every 160' implies full chunks)
        print(f"Skipping subject {pid} with only {total_trials} trials")
        continue

    # Extract all data for this subject
    all_x = grp[['action', 'reward']].to_numpy().astype(float)
    all_y = grp[['action_n']].to_numpy().astype(int)

    # Slice into chunks
    for i in range(n_segments):
        start_idx = i * seg_len
        end_idx = start_idx + seg_len

        x_chunk = all_x[start_idx:end_idx]
        y_chunk = all_y[start_idx:end_idx]

        xs_list_suth.append(x_chunk)
        ys_list_suth.append(y_chunk)

# Stack to (n_total_segments, n_trials, feat_dim)
xs_suth_stacked = np.stack(xs_list_suth, axis=0)
ys_suth_stacked = np.stack(ys_list_suth, axis=0)

# Transpose to (n_trials, n_sessions, feat_dim) -> (160, N, 2) as requested
xs_suth = xs_suth_stacked.transpose(1, 0, 2)
ys_suth = ys_suth_stacked.transpose(1, 0, 2)

print(f"xs_suth.shape = {xs_suth.shape}")
print(f"ys_suth.shape = {ys_suth.shape}")

xs_suth.shape = (160, 938, 2)
ys_suth.shape = (160, 938, 1)


In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1c5Wkszr_wp04GAyz-g727EullL7mT8r4'
download_url = f'https://drive.google.com/uc?id={file_id}'

# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
barnby_data = pd.read_csv(output_file)

# Display the first few rows
barnby_data.head()

Downloading...
From: https://drive.google.com/uc?id=1c5Wkszr_wp04GAyz-g727EullL7mT8r4
To: /content/downloaded_new_file.csv
100%|██████████| 2.24M/2.24M [00:00<00:00, 26.2MB/s]


,ID,Trial,Selection,Block,Correct,Response,Age
0,54240de1fdf99b691fb383a8,1,urn3.png,Block 1,0,-5,57
1,54240de1fdf99b691fb383a8,2,urn2.png,Block 1,0,-5,57
2,54240de1fdf99b691fb383a8,3,urn1.png,Block 1,1,10,57
3,54240de1fdf99b691fb383a8,4,urn1.png,Block 1,1,10,57
4,54240de1fdf99b691fb383a8,5,urn1.png,Block 1,1,10,57


In [ ]:
# 1. 把 Block 2 的 trial 从 1-30 变成 31-60
barnby_data.loc[barnby_data['Block'] == 'Block 2', 'Trial'] += 30

# 2. 将 Selection 里的图片名称重新编码成 1, 2, 3
barnby_data['Selection'] = barnby_data['Selection'].replace({
    'urn1.png': 1,
    'urn2.png': 2,
    'urn3.png': 3
})

# 3. 将 Response 重新编码（-5 变成 0，假设正收益 10 变成 1）
barnby_data['Response'] = barnby_data['Response'].replace({-5: 0, 10: 1})

# 查看处理后的结果
barnby_data.head()


,ID,Trial,Selection,Block,Correct,Response,Age
0,54240de1fdf99b691fb383a8,1,3,Block 1,0,0,57
1,54240de1fdf99b691fb383a8,2,2,Block 1,0,0,57
2,54240de1fdf99b691fb383a8,3,1,Block 1,1,1,57
3,54240de1fdf99b691fb383a8,4,1,Block 1,1,1,57
4,54240de1fdf99b691fb383a8,5,1,Block 1,1,1,57


In [ ]:
import numpy as np

# 1. 确保数据按被试和试次顺序排序
barnby_data = barnby_data.sort_values(['ID', 'Trial']).reset_index(drop=True)

# 2. 生成下一步的动作 (action_n)，并在每个被试最后 trial 填补 -1
barnby_data['action_n'] = barnby_data.groupby('ID')['Selection'].shift(-1).fillna(-1).astype(int)

xs_list_barnby = []
ys_list_barnby = []
seg_len = 160

# 3. 按被试遍历并构造 padding
for pid, grp in barnby_data.groupby('ID'):
    x_seq = grp[['Selection', 'Response']].to_numpy().astype(float)
    y_seq = grp[['action_n']].to_numpy().astype(int)

    # 计算需要 pad 的长度 (期望是 160 - 60 = 100)
    pad_len = seg_len - len(x_seq)

    if pad_len > 0:
        # xs pad with 0.
        x_seq = np.pad(x_seq, pad_width=((0, pad_len), (0, 0)), constant_values=0.)
        # ys pad with -1
        y_seq = np.pad(y_seq, pad_width=((0, pad_len), (0, 0)), constant_values=-1)
    elif pad_len < 0:
        # 截断（如果多于160的情况，以防万一）
        x_seq = x_seq[:seg_len]
        y_seq = y_seq[:seg_len]

    xs_list_barnby.append(x_seq)
    ys_list_barnby.append(y_seq)

# 4. 堆叠成三维数组 (n_sessions, 160, feat_dim)
xs_barnby_stacked = np.stack(xs_list_barnby, axis=0)
ys_barnby_stacked = np.stack(ys_list_barnby, axis=0)

# 5. 转置为模型需要的维度: (n_trials, n_sessions, feat_dim) -> (160, N, 2/1)
xs_barnby = xs_barnby_stacked.transpose(1, 0, 2)
ys_barnby = ys_barnby_stacked.transpose(1, 0, 2)

print(f"xs_barnby.shape = {xs_barnby.shape}")
print(f"ys_barnby.shape = {ys_barnby.shape}")

xs_barnby.shape = (160, 697, 2)
ys_barnby.shape = (160, 697, 1)


In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1eUdMZ1_HHo7Uw5LcZDVGGV_aRNVxQT14'
download_url = f'https://drive.google.com/uc?id={file_id}'


# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
shahar_data = pd.read_csv(output_file)

# Display the first few rows
shahar_data.head()

Downloading...
From: https://drive.google.com/uc?id=1eUdMZ1_HHo7Uw5LcZDVGGV_aRNVxQT14
To: /content/downloaded_new_file.csv
100%|██████████| 2.82M/2.82M [00:00<00:00, 75.0MB/s]


,subject,measurment,trial,choice1,reward,transition
0,1,1,3,2,0,0
1,1,1,4,1,0,1
2,1,1,5,1,0,1
3,1,1,6,2,0,0
4,1,1,7,1,0,0


In [ ]:
import numpy as np

# 1. 创建唯一的被试ID (基于 subject 和 measurment)
shahar_data['unique_id'] = shahar_data.groupby(['subject', 'measurment']).ngroup()

# 2. 确保数据按被试和试次顺序排序
shahar_data = shahar_data.sort_values(['unique_id', 'trial']).reset_index(drop=True)

# 3. 生成下一步的动作 (action_n)，并在每个被试最后 trial 填补 -1
shahar_data['action_n'] = shahar_data.groupby('unique_id')['choice1'].shift(-1).fillna(-1).astype(int)

xs_list_sha = []
ys_list_sha = []
seg_len = 201

# 4. 按 unique_id 遍历并构造 padding
for pid, grp in shahar_data.groupby('unique_id'):
    # 特征包含 choice1, reward, transition 三列
    x_seq = grp[['choice1', 'reward', 'transition']].to_numpy().astype(float)
    y_seq = grp[['action_n']].to_numpy().astype(int)

    # 计算需要 pad 的长度
    pad_len = seg_len - len(x_seq)

    if pad_len > 0:
        # xs pad with 0.
        x_seq = np.pad(x_seq, pad_width=((0, pad_len), (0, 0)), constant_values=0.)
        # ys pad with -1
        y_seq = np.pad(y_seq, pad_width=((0, pad_len), (0, 0)), constant_values=-1)
    elif pad_len < 0:
        # 截断（如果多于201的情况，以防万一）
        x_seq = x_seq[:seg_len]
        y_seq = y_seq[:seg_len]

    xs_list_sha.append(x_seq)
    ys_list_sha.append(y_seq)

# 5. 堆叠成三维数组 (n_sessions, 201, feat_dim)
xs_sha_stacked = np.stack(xs_list_sha, axis=0)
ys_sha_stacked = np.stack(ys_list_sha, axis=0)

# 6. 转置为模型需要的维度: (n_trials, n_sessions, feat_dim) -> (201, N, 3 / 1)
xs_sha = xs_sha_stacked.transpose(1, 0, 2)
ys_sha = ys_sha_stacked.transpose(1, 0, 2)

print(f"xs_sha.shape = {xs_sha.shape}")
print(f"ys_sha.shape = {ys_sha.shape}")


xs_sha.shape = (201, 1108, 3)
ys_sha.shape = (201, 1108, 1)


In [ ]:
import gdown
import pandas as pd

# Extract file_id from the link: https://drive.google.com/file/d/1kO6fM9v6ECcYWt3DYkbgWZaRgvmABvUX/view?usp=drive_link
file_id = '1Qqrs20pf45SyB6HatfOnXFf4qwG53zFe'
download_url = f'https://drive.google.com/uc?id={file_id}'


# Download and save
output_file = 'downloaded_new_file.csv'
gdown.download(download_url, output_file, quiet=False)

# Read CSV data
eisenberg_data = pd.read_csv(output_file)

# Display the first few rows
eisenberg_data.head()

Downloading...
From: https://drive.google.com/uc?id=1Qqrs20pf45SyB6HatfOnXFf4qwG53zFe
To: /content/downloaded_new_file.csv
100%|██████████| 7.30M/7.30M [00:00<00:00, 44.8MB/s]


,Unnamed: 0,exp_stage,feedback,feedback_last,stage_second,stage_transition,stim_selected_first,trial_num
0,two_stage_decision_s000_000,practice,1.0,NaN,2,frequent,1,0
1,two_stage_decision_s000_001,practice,1.0,1.0,1,frequent,0,1
2,two_stage_decision_s000_002,practice,1.0,1.0,1,frequent,0,2
3,two_stage_decision_s000_003,practice,0.0,1.0,2,frequent,1,3
4,two_stage_decision_s000_004,practice,1.0,0.0,1,frequent,0,4


In [ ]:
import numpy as np
import pandas as pd

# 1. 提取被试编号
eisenberg_data['subject'] = eisenberg_data['Unnamed: 0'].str.split('_').str[3]

# 2. 扔掉 practice 的行
eisenberg_data = eisenberg_data[eisenberg_data['exp_stage'] != 'practice'].reset_index(drop=True)

# 3. 把 stage_transition 里的 frequent 编码成 0，另一个编码成 1
eisenberg_data['stage_transition'] = np.where(eisenberg_data['stage_transition'] == 'frequent', 0, 1)

# 将所需的特征列转为合适类型，处理第一试次可能的 NaN
eisenberg_data['stim_selected_first'] = eisenberg_data['stim_selected_first'].astype(int)
eisenberg_data['feedback_last'] = eisenberg_data['feedback_last'].fillna(0).astype(float)

# 4. 生成 unique_id 并排序
eisenberg_data['unique_id'] = eisenberg_data.groupby('subject').ngroup()
eisenberg_data = eisenberg_data.sort_values(['unique_id', 'trial_num']).reset_index(drop=True)

# 5. 生成 action_n
eisenberg_data['action_n'] = eisenberg_data.groupby('unique_id')['stim_selected_first'].shift(-1).fillna(-1).astype(int)

xs_list_eisen = []
ys_list_eisen = []
seg_len = 201

# 6. 按 unique_id 遍历并构造 padding
for pid, grp in eisenberg_data.groupby('unique_id'):
    x_seq = grp[['stim_selected_first', 'feedback_last', 'stage_transition']].to_numpy().astype(float)
    y_seq = grp[['action_n']].to_numpy().astype(int)

    pad_len = seg_len - len(x_seq)

    if pad_len > 0:
        # xs pad with 0.
        x_seq = np.pad(x_seq, pad_width=((0, pad_len), (0, 0)), constant_values=0.)
        # ys pad with -1
        y_seq = np.pad(y_seq, pad_width=((0, pad_len), (0, 0)), constant_values=-1)
    elif pad_len < 0:
        # 截断（如果多于201的情况，以防万一）
        x_seq = x_seq[:seg_len]
        y_seq = y_seq[:seg_len]

    xs_list_eisen.append(x_seq)
    ys_list_eisen.append(y_seq)

# 7. 堆叠成三维数组
xs_eisen_stacked = np.stack(xs_list_eisen, axis=0)
ys_eisen_stacked = np.stack(ys_list_eisen, axis=0)

# 8. 转置为模型需要的维度: (n_trials, n_sessions, feat_dim)
xs_eisen = xs_eisen_stacked.transpose(1, 0, 2)
ys_eisen = ys_eisen_stacked.transpose(1, 0, 2)

print(f"xs_eisen.shape = {xs_eisen.shape}")
print(f"ys_eisen.shape = {ys_eisen.shape}")


xs_eisen.shape = (201, 522, 3)
ys_eisen.shape = (201, 522, 1)
